# Amaliy mashgʻulot 5. Statik vektorlashtirish: BoW, TF-IDF, Word2Vec, FastText

5-laboratoriyada «hujjat — soʻz» matritsasini, oʻyinchoq korpusda skip-gram oʻqitilishini va FastText belgi n-grammalarini koʻrdingiz. Bu yerda hammasini kodda takrorlaymiz:

1. soʻzlar qopi (BoW) va TF-IDF ni qoʻlda quramiz, `scikit-learn` bilan solishtiramiz va hujjatlarni kosinus oʻxshashligi boʻyicha taqqoslaymiz;
2. salbiy namunali skip-gram (SGNS) ni `numpy` da yozamiz va 38 gapli korpusda oʻqitamiz;
3. `gensim` kutubxonasidagi `Word2Vec` ni oʻsha korpusda oʻqitamiz, qoʻshnilarni solishtiramiz va «shoh − erkak + ayol» analogiyasini koʻramiz;
4. FastText ni tahlil qilamiz: belgi n-grammalari va korpusda boʻlmagan shakllar uchun vektorlar («kitoblarimizdan»).

*«NLP algoritmlari» kursining 5-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

In [ ]:
# gensim Colab da oldindan oʻrnatilmagan: oʻrnatamiz (taxminan 30 soniya)
!pip install -q gensim

## 1. Soʻzlar qopi (BoW) va TF-IDF

Hujjat $|V|$ uzunlikdagi vektorga aylanadi ($|V|$ — lugʻat hajmi). **Soʻzlar qopi**da har bir koordinata soʻz hujjatda necha marta uchraganini bildiradi, soʻzlar tartibi yoʻqoladi. Koʻp hujjatlarda uchraydigan soʻzlar (masalan, «va» bogʻlovchisi) mazmun haqida kam maʼlumot beradi, shuning uchun **TF-IDF** chastotani teskari hujjat chastotasiga koʻpaytiradi:

$$\text{tfidf}(w,d)=\text{tf}(w,d)\cdot\text{idf}(w),\qquad \text{idf}(w)=\ln\frac{1+N}{1+\text{df}(w)}+1,$$

bu yerda $N$ — hujjatlar soni, $\text{df}(w)$ — $w$ soʻzi nechta hujjatda borligi. Bu scikit-learn varianti (`smooth_idf=True`); koʻpaytirgandan keyin har bir satr-vektor oʻz uzunligiga (L2-normasiga) boʻlinadi.

In [ ]:
import re
import numpy as np
import pandas as pd

docs = [
    "Mushuk divanda uxlaydi, mushuk divanni yaxshi koʻradi.",
    "It hovlida uxlaydi.",
    "Mushuk va it hovlida oʻynaydi.",
    "Onam oshxonada shoʻrva pishiradi.",
    "Otam oshxonada choy damlaydi.",
    "Oshxonadan shoʻrva va choy hidi keladi.",
]

def tokenize(text):
    # soʻzlar: harf va raqamlar ketma-ketligi, kichik harflarda
    return re.findall(r"\w+", text.lower())

tok_docs = [tokenize(d) for d in docs]
vocab = sorted({w for d in tok_docs for w in d})
w2i = {w: i for i, w in enumerate(vocab)}
N, V = len(docs), len(vocab)
names = ["d" + str(i + 1) for i in range(N)]

# BoW: N x V matritsa, katakda — soʻz hujjatda necha marta uchragani
counts = np.zeros((N, V))
for i, d in enumerate(tok_docs):
    for w in d:
        counts[i, w2i[w]] += 1

# IDF scikit-learn dagidek (smooth_idf=True)
df = (counts > 0).sum(axis=0)
idf = np.log((1 + N) / (1 + df)) + 1
tfidf = counts * idf
tfidf = tfidf / np.linalg.norm(tfidf, axis=1, keepdims=True)   # satrlarni L2-normalash

for name, d in zip(names, docs):
    print(name, d)
table = pd.DataFrame(counts.T.astype(int), index=vocab, columns=names)
table["df"] = df
table["idf"] = idf.round(2)
print()
print(table.to_string())

print("TF-IDF (ustunlar — hujjatlar):")
print(pd.DataFrame(tfidf, index=names, columns=vocab).round(2).T.to_string())
print()
for i in range(N):
    top = np.argsort(-tfidf[i], kind="stable")[:3]
    print(names[i], "eng vaznli soʻzlar:", ", ".join(f"{vocab[j]} ({tfidf[i, j]:.2f})" for j in top))

Xuddi shu matritsalarni scikit-learn ham beradi. Lugʻatlar mos kelishi uchun vektorlashtirgichlarga oʻzimizning `tokenize` funksiyamizni beramiz (odatda sklearn bir harfli soʻzlarni tashlab yuboradi).


Hujjatlarning **kosinus oʻxshashligi**: $\cos(a,b)=\dfrac{a\cdot b}{\|a\|\,\|b\|}$. BoW va TF-IDF uchun oʻxshashlik matritsalarini solishtiramiz.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
X_bow = cv.fit_transform(docs).toarray()
tv = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
X_tfidf = tv.fit_transform(docs).toarray()

print("lugʻat mos keladi: ", list(cv.get_feature_names_out()) == vocab)
print("BoW mos keladi:     ", np.array_equal(X_bow, counts))
print("IDF mos keladi:     ", np.allclose(tv.idf_, idf))
print("TF-IDF mos keladi:  ", np.allclose(X_tfidf, tfidf))

import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

def cosine_matrix(X):
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    return Xn @ Xn.T

S_bow, S_tfidf = cosine_matrix(counts), cosine_matrix(tfidf)
print("sklearn bilan mos keladi:", np.allclose(S_tfidf, cosine_similarity(X_tfidf)))
print("\nBoW:")
print(pd.DataFrame(S_bow, index=names, columns=names).round(2).to_string())
print("\nTF-IDF:")
print(pd.DataFrame(S_tfidf, index=names, columns=names).round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, S, title in zip(axes, [S_bow, S_tfidf], ["BoW", "TF-IDF"]):
    ax.imshow(S, cmap="Blues", vmin=0, vmax=1)
    for i in range(N):
        for j in range(N):
            ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if S[i, j] > 0.6 else "black")
    ax.set_xticks(range(N), names)
    ax.set_yticks(range(N), names)
    ax.set_title("Hujjatlarning kosinus oʻxshashligi, " + title)
plt.tight_layout()
plt.show()

d4–d5 juftligiga qarang («Onam oshxonada shoʻrva pishiradi» va «Otam oshxonada choy damlaydi»): BoW da ularning yagona umumiy soʻzi — «oshxonada», oʻxshashlik 0,25. TF-IDF da «oshxonada» ikki hujjatda uchraydi va noyob «onam», «otam», «pishiradi», «damlaydi» soʻzlaridan kichikroq vazn oladi, shuning uchun oʻxshashlik 0,20 ga tushadi. d6 dagi «oshxonadan» esa «oshxonada» bilan umuman bogʻlanmagan: BoW uchun bu ikki xil soʻz, xuddi «divanda» va «divanni» kabi. Umumiy soʻzi yoʻq juftliklar (d1–d4, d2–d5 va boshqalar) ikkala modelda ham nolga teng: bu modellar «mushuk» va «it» maʼnoda yaqinligini bilmaydi. Agglyutinativ tilda bu muammo ayniqsa kuchli. Bu masalalarni soʻz vektorlari hal qiladi.

## 2. Salbiy namunali Word2Vec skip-gram: numpy da noldan

Distributiv semantika gʻoyasi: oʻxshash kontekstlarda uchraydigan soʻzlar maʼnoda yaqin. **Skip-gram** markaziy soʻz $c$ ni olib, uning atrofidagi oynadagi $o$ soʻzlarini bashorat qilishni oʻrganadi. Har bir soʻzning ikki vektori bor: $v_w$ (markaz sifatida) va $u_w$ (kontekst sifatida). Butun lugʻat boʻyicha softmax oʻrniga **salbiy namunalar** (negative sampling) ishlatamiz: haqiqiy $(c, o)$ juftlik uchun $K$ ta tasodifiy soʻz $n_1,\dots,n_K$ olinadi va quyidagi funksiya minimallashtiriladi:

$$L = -\log\sigma(u_o^\top v_c) - \sum_{k=1}^{K}\log\sigma(-u_{n_k}^\top v_c),\qquad \sigma(x)=\frac{1}{1+e^{-x}}.$$

Tasodifiy soʻzlar $P(w)\propto \text{freq}(w)^{0.75}$ ehtimol bilan tanlanadi: 0,75 darajasi kam uchraydigan soʻzlarni biroz koʻtaradi. Quyidagi korpus — oila, hayvonlar, taomlar va oʻquv qurollari haqida 38 ta qisqa gap.


Avval korpusni sirpanuvchi oyna bilan (markaz, kontekst) juftliklariga ajratamiz — laboratoriyadagi animatsiya kabi.

In [ ]:
from collections import Counter

corpus = """
onam uyda kitob oʻqiydi
otam uyda gazeta oʻqiydi
buvim nevaralariga ertak aytadi
akam uyda xat yozadi
opam buvimga xat yozadi
onam va otam ertalab ishlaydi
akam va opam ertalab oʻqiydi
otam onam bilan telefonda gaplashadi
mushuk hovlida uxlaydi
it hovlida uxlaydi
it hovlida yuguradi
mushuk maysada yuguradi
sigir yaylovda oʻt yeydi
qoʻy yaylovda oʻt yeydi
echki yaylovda suv ichadi
qoʻy daryo boʻyida suv ichadi
sigir daryo boʻyida suv ichadi
mushuk iliq sut ichadi
onam mazali shoʻrva pishiradi
buvim mazali palov pishiradi
otam issiq non oladi
opam issiq somsa oladi
akam issiq shoʻrvani non bilan yeydi
tushlikda palov va non yeymiz
kechqurun shoʻrva va somsa yeymiz
issiq non stol ustida turibdi
kitob javonda turibdi
kitoblar javonda turibdi
kitobni sumkaga soldim
kitobimiz juda qiziq
kitoblarimiz sumkada turibdi
kitobdan sheʼr oʻqidim
daftar va ruchka sumkada turibdi
daftarni stolga qoʻydim
daftarlarimiz sumkada turibdi
ruchka bilan daftarga yozdim
qalamni sumkaga soldim
akam kitoblarni javonga qoʻydi
"""

sents = [tokenize(line) for line in corpus.strip().splitlines()]
freq = Counter(w for s in sents for w in s)
words = sorted(freq)                      # korpus lugʻati
idx = {w: i for i, w in enumerate(words)}
print(len(sents), "gap,", sum(freq.values()), "soʻz qoʻllanishi,", len(words), "xil soʻz")
print("eng koʻp uchraydiganlar:", freq.most_common(8))

WINDOW = 2

def context_positions(sent, i, window):
    return [j for j in range(max(0, i - window), min(len(sent), i + window + 1)) if j != i]

# oyna birinchi gap boʻylab sirpanadi
s0 = sents[0]
for i, c in enumerate(s0):
    print(f"{c:>12} → {[s0[j] for j in context_positions(s0, i, WINDOW)]}")

pairs = np.array([(idx[s[i]], idx[s[j]]) for s in sents for i in range(len(s))
                  for j in context_positions(s, i, WINDOW)])
print("\njami (markaz, kontekst) juftliklari:", len(pairs))

# salbiy namunalar uchun taqsimot: freq^0.75
f = np.array([freq[w] for w in words], dtype=float)
noise = f ** 0.75 / (f ** 0.75).sum()
for w in [w for w, _ in freq.most_common(3)] + [words[int(np.argmin(f))]]:
    print(f"   {w:<10} chastota {f[idx[w]] / f.sum():.3f}   P_noise {noise[idx[w]]:.3f}")

Endi oʻqitish. Har bir juftlik uchun haqiqiy kontekst va $K=5$ ta salbiy soʻz boʻyicha $\sigma(u^\top v)$ ni hisoblaymiz; ball boʻyicha gradient $\sigma(u^\top v) - \text{belgi}$ ga teng (haqiqiy juftlikda belgi 1, tasodifiylarida 0). SGD qadami $v_c$ ni $u_o$ ga yaqinlashtiradi va $u_{n_k}$ dan uzoqlashtiradi. 372 juftlik boʻyicha 100 epoxa taxminan bir soniya oladi.


Model nimani oʻrgandi? $\sigma(u_o^\top v_c)$ — juftlik «haqiqiy» ekanligi ehtimolining model bahosi. Korpusdagi va korpusda yoʻq juftliklarni solishtiramiz, soʻng $v$ vektorlarining kosinusi boʻyicha eng yaqin qoʻshnilarni topamiz.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def train_sgns(pairs, V, dim=10, neg=5, lr=0.05, epochs=100, seed=0):
    rng = np.random.default_rng(seed)
    W_in = (rng.random((V, dim)) - 0.5) / dim    # v: markaziy soʻzlar vektorlari (embedding sifatida shularni olamiz)
    W_out = np.zeros((V, dim))                   # u: kontekst soʻzlari vektorlari
    labels = np.zeros(neg + 1)
    labels[0] = 1                                # 1 ta haqiqiy juftlik + neg ta tasodifiy
    history = []
    for ep in range(epochs):
        negs = rng.choice(V, size=(len(pairs), neg), p=noise)
        total = 0.0
        for n in rng.permutation(len(pairs)):
            c, o = pairs[n]
            ids = np.concatenate(([o], negs[n]))
            v, u = W_in[c], W_out[ids]
            p = sigmoid(u @ v)
            g = p - labels                       # yoʻqotishning u·v ballari boʻyicha gradienti
            total += -np.log(p[0] + 1e-12) - np.log(1 - p[1:] + 1e-12).sum()
            W_in[c] -= lr * g @ u
            np.add.at(W_out, ids, -lr * np.outer(g, v))
        history.append(total / len(pairs))
        if ep % 20 == 0 or ep == epochs - 1:
            print(f"epoxa {ep:3d}   oʻrtacha yoʻqotish {history[-1]:.3f}")
    return W_in, W_out, history

W_in, W_out, history = train_sgns(pairs, len(words))

In [ ]:
def pair_prob(a, b):
    return sigmoid(W_out[idx[b]] @ W_in[idx[a]])

print("korpusdagi juftliklar:")
for a, b in [("mushuk", "sut"), ("shoʻrva", "mazali"), ("sigir", "yaylovda"), ("daftar", "ruchka")]:
    print(f"   σ(u_{b} · v_{a}) = {pair_prob(a, b):.2f}")
print("korpusda yoʻq juftliklar:")
for a, b in [("mushuk", "gazeta"), ("shoʻrva", "javonda"), ("sigir", "xat"), ("daftar", "daryo")]:
    print(f"   σ(u_{b} · v_{a}) = {pair_prob(a, b):.2f}")

E = W_in / np.linalg.norm(W_in, axis=1, keepdims=True)

def neighbours(E, word, k=5):
    sims = E @ E[idx[word]]
    order = [j for j in np.argsort(-sims) if j != idx[word]][:k]
    return [(words[j], round(float(sims[j]), 2)) for j in order]

print()
for q in ["mushuk", "shoʻrva", "non", "onam", "daftar"]:
    print(f"{q:>8}:", neighbours(E, q))

Bunday kichik korpusda qoʻshnilar shovqinli, lekin qonuniyat koʻrinadi: hayvonlar hayvonlar va ularning harakatlari yonida («it», «yuguradi», «maysada»), taomlar taomlar yonida («shoʻrva» — «palov», «non» — «somsa»), oʻquv qurollari esa bir-biri va «sumkada» yonida. Vektorlarni bosh komponentalar usuli (PCA) bilan tekislikda chizamiz.

In [ ]:
groups = {
    "oila": ["onam", "otam", "buvim", "akam", "opam"],
    "hayvonlar": ["mushuk", "it", "sigir", "qoʻy", "echki"],
    "taom": ["shoʻrva", "palov", "non", "somsa", "sut"],
    "oʻquv qurollari": ["kitob", "kitoblar", "daftar", "ruchka"],
}

# SVD orqali PCA: ikki bosh komponentaga proyeksiya
Xc = W_in - W_in.mean(axis=0)
U_, S_, Vt = np.linalg.svd(Xc, full_matrices=False)
P2 = Xc @ Vt[:2].T
print("ikki komponentadagi dispersiya ulushi:", round(float((S_[:2] ** 2).sum() / (S_ ** 2).sum()), 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(history, color="#2B55C9")
axes[0].set_xlabel("epoxa")
axes[0].set_ylabel("juftlikka oʻrtacha yoʻqotish")
axes[0].set_title("SGNS ni oʻqitish")
colors = ["#2B55C9", "#D9532C", "#17907E", "#8E44AD"]
for (g, ws), col in zip(groups.items(), colors):
    pts = np.array([P2[idx[w]] for w in ws])
    axes[1].scatter(pts[:, 0], pts[:, 1], color=col, label=g)
    for w, (x, y) in zip(ws, pts):
        axes[1].annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[1].legend(fontsize=8)
axes[1].set_title("Soʻz vektorlari, PCA")
plt.tight_layout()
plt.show()

## 3. gensim dagi Word2Vec

`gensim` kutubxonasi xuddi shu algoritmni (`sg=1` — skip-gram, `negative=5` — beshta salbiy namuna) C tilida, oʻqitish qadamini kamaytirib va koʻp uchraydigan soʻzlarni siyraklashtirib amalga oshiradi. Kichkina korpusda siyraklashtirishni oʻchiramiz (`sample=0`), epoxalarni esa koʻproq olamiz. `workers=1` va `seed` natijani takrorlanadigan qiladi.

In [ ]:
import time
from gensim.models import Word2Vec

t0 = time.time()
w2v = Word2Vec(sents, vector_size=20, window=2, min_count=1, sg=1, negative=5,
               sample=0, epochs=300, seed=1, workers=1)
print(f"oʻqitish vaqti {time.time() - t0:.1f} s, lugʻat {len(w2v.wv)}, oʻlcham {w2v.wv.vector_size}")

rows = []
for q in ["mushuk", "shoʻrva", "non", "onam", "daftar"]:
    ours = [w for w, _ in neighbours(E, q)]
    lib = [w for w, _ in w2v.wv.most_similar(q, topn=5)]
    rows.append((q, ", ".join(ours), ", ".join(lib), len(set(ours) & set(lib))))
print(pd.DataFrame(rows, columns=["soʻz", "bizning SGNS", "gensim", "umumiy"]).to_string(index=False))

**Analogiyalar.** Mashhur «shoh − erkak + ayol ≈ malika» milliardlab soʻzda oʻqitilgan vektorlarda ishlaydi: u yerda «ayol − erkak» ayirmasi koʻp juftliklar uchun deyarli bir xil. 150 soʻzli korpusimizda analogiya chiqmaydi, shuning uchun (laboratoriyadagidek) **qoʻlda berilgan** kichik vektorlar toʻplamini olamiz. Uch oʻq: «hukmdorlik», «jins», «yosh». gensim dagi `most_similar(positive=..., negative=...)` usuli vektorlar yigʻindisiga eng yaqin soʻzni qidiradi.


Grafikda «erkak → ayol», «shoh → malika», «ota → ona» strelkalari parallel: bu «jins yoʻnalishi». `most_similar` kirish soʻzlarining oʻzini javobdan chiqarib tashlaydi.

In [ ]:
from gensim.models import KeyedVectors

an_words = ["shoh", "malika", "erkak", "ayol", "ota", "ona", "oʻgʻil", "qiz"]
#          hukmdor. jins  yosh
an_vectors = [[0.9,  0.8, 0.7],   # shoh
              [0.9, -0.8, 0.7],   # malika
              [0.1,  0.8, 0.5],   # erkak
              [0.1, -0.8, 0.5],   # ayol
              [0.2,  0.7, 0.9],   # ota
              [0.2, -0.7, 0.9],   # ona
              [0.1,  0.7, 0.1],   # oʻgʻil
              [0.1, -0.7, 0.1]]   # qiz
queries = [(["shoh", "ayol"], ["erkak"]),
           (["ota", "ayol"], ["erkak"]),
           (["oʻgʻil", "ayol"], ["erkak"])]
ARROWS = [("erkak", "ayol"), ("shoh", "malika"), ("ota", "ona")]

kv = KeyedVectors(vector_size=3)
kv.add_vectors(an_words, np.array(an_vectors, dtype=np.float32))

for pos, neg in queries:
    best = kv.most_similar(positive=pos, negative=neg, topn=2)
    expr = f"{pos[0]} − {neg[0]} + {pos[1]}"
    print(f"{expr:<28} ≈ {best[0][0]:<10} (cos {best[0][1]:.2f});  ikkinchi oʻrin: {best[1][0]} ({best[1][1]:.2f})")

# xuddi shu qoʻlda: vektorlar yigʻindisi va barcha soʻzlarga kosinus
A = np.array(an_vectors)
An = A / np.linalg.norm(A, axis=1, keepdims=True)
pos, neg = queries[0]
target = An[an_words.index(pos[0])] - An[an_words.index(neg[0])] + An[an_words.index(pos[1])]
sims = An @ target / np.linalg.norm(target)
cand = [(an_words[j], round(float(sims[j]), 2)) for j in np.argsort(-sims) if an_words[j] not in pos + neg]
print("qoʻlda:", cand[:2])

fig, ax = plt.subplots(figsize=(5, 4))
# «jins» va «yosh» oʻqlarini chizamiz; nuqta oʻlchami — «hukmdorlik»
ax.scatter(A[:, 1], A[:, 2], s=40 + 200 * A[:, 0], color="#2B55C9")
for w, (r, g, a) in zip(an_words, A):
    ax.annotate(w, (g, a), fontsize=9, xytext=(0, 9), textcoords="offset points", ha="center")
for a, b in ARROWS:
    pa, pb = A[an_words.index(a)], A[an_words.index(b)]
    ax.annotate("", xy=(pb[1], pb[2]), xytext=(pa[1], pa[2]), arrowprops=dict(arrowstyle="->", color="#D9532C"))
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(0, 1.1)
ax.set_xlabel("«jins» oʻqi")
ax.set_ylabel("«yosh» oʻqi")
ax.set_title("Qoʻlda berilgan vektorlar: «jins» — bitta yoʻnalish")
plt.tight_layout()
plt.show()

## 4. FastText: belgi n-grammalari va notanish soʻzlar

Word2Vec da korpusda boʻlmagan soʻz uchun vektor yoʻq: u uchun «kitoblarimizdan» shunchaki notanish satr. **FastText** soʻzni belgi n-grammalari toʻplami sifatida ifodalaydi: soʻzga `<` va `>` chegaralari qoʻshiladi va $n=3\ldots5$ da «kitob» quyidagilarni beradi: `<ki`, `kit`, `ito`, `tob`, `ob>`, `<kit`, `kito`, `itob`, `tob>`, `<kito`, `kitob`, `itob>`. Soʻz vektori n-grammalar vektorlaridan yigʻiladi (gensim da — oʻrtacha; korpusdagi soʻz uchun uning oʻz vektori ham qoʻshiladi). Notanish shakl uchun vektor tanish boʻlaklardan yigʻiladi. Qoʻshimchalar zanjiri uzun boʻlgan oʻzbek tili uchun bu ayniqsa muhim: bitta oʻzakdan oʻnlab shakllar hosil boʻladi va ularning koʻpi kichik korpusda uchramaydi.

In [ ]:
from gensim.models.fasttext import compute_ngrams

def char_ngrams(word, min_n=3, max_n=5):
    w = "<" + word + ">"
    return [w[i:i + n] for n in range(min_n, max_n + 1) for i in range(len(w) - n + 1)]

word = "kitoblarimizdan"
grams = char_ngrams(word)
print(word, "→", len(grams), "n-gramma:")
print("  ", grams)
print("gensim bilan mos keladi:", sorted(grams) == sorted(compute_ngrams(word, 3, 5)))

# korpusdagi qaysi soʻzlar u bilan eng koʻp n-gramma baham koʻradi (Jaccard koeffitsiyenti)
g = set(grams)
share = sorted(((len(g & set(char_ngrams(w))) / len(g | set(char_ngrams(w))), w) for w in words), reverse=True)
print("n-grammalar boʻyicha eng yaqinlar:", [(w, round(j, 2)) for j, w in share[:5]])

gensim da FastText ni oʻsha korpusda oʻqitamiz va korpusda yoʻq shakllar uchun vektor soʻraymiz. Word2Vec bu soʻzlarda `KeyError` bilan toʻxtaydi.


gensim haqiqatan ham n-grammalarni qoʻshishini tekshiramiz: notanish soʻz vektorini oʻzimiz `ft.wv.vectors_ngrams` dagi n-gramma vektorlarining oʻrtachasi sifatida yigʻamiz (satr raqami — n-gramma xeshi, `ft_ngram_hashes`).

In [ ]:
from gensim.models import FastText

ft = FastText(sents, vector_size=20, window=2, min_count=1, sg=1, negative=5, sample=0,
              epochs=300, seed=1, workers=1, min_n=3, max_n=5, bucket=20000)

for w in ["kitoblarimizdan", "daftarlarimizdan", "mushuklar"]:
    print(w, "| korpusda:", w in ft.wv.key_to_index)
    try:
        w2v.wv[w]
    except KeyError:
        print("   Word2Vec: KeyError — vektor yoʻq")
    print("   FastText:", [(x, round(s, 2)) for x, s in ft.wv.most_similar(w, topn=4)])

from gensim.models.fasttext import ft_ngram_hashes

w = "kitoblarimizdan"
hashes = ft_ngram_hashes(w, ft.wv.min_n, ft.wv.max_n, ft.wv.bucket)
manual = ft.wv.vectors_ngrams[hashes].mean(axis=0)
print(len(hashes), "ta satr n-grammalar jadvalidan, birinchi raqamlar:", hashes[:5])
print("qoʻlda yigʻilgan vektor ft.wv[w] bilan mos keladi:", np.allclose(manual, ft.wv[w], atol=1e-6))

Belgi n-grammalarining teskari tomoni ham bor: «suvenir» «suv» bilan `<su`, `suv`, `<suv` boshlanishini baham koʻradi, lekin maʼnoda ular boshqa-boshqa soʻzlar. Grafikda notanish shakllarning «oʻz» soʻzlariga oʻxshashligini va shu soxta oʻxshashlikni solishtiramiz. Imlo xatosi («kitop») ham toʻgʻri soʻz yonida qoladi.

In [ ]:
pairs_plot = [("kitoblarimizdan", "kitob"), ("kitoblarda", "kitob"), ("daftarlarda", "daftar"),
              ("mushuklar", "mushuk"), ("kitop", "kitob"), ("suvenir", "suv")]
FALSE_FRIEND = "suvenir"

sims = [float(ft.wv.similarity(a, b)) for a, b in pairs_plot]
labels = [f"{a}\n~ {b}" for a, b in pairs_plot]
for (a, b), s in zip(pairs_plot, sims):
    print(f"{a:>17} ~ {b:<8} cos = {s:.2f}   (korpusda: {a in ft.wv.key_to_index})")

fig, ax = plt.subplots(figsize=(9, 3.6))
cols = ["#D9532C" if a == FALSE_FRIEND else "#17907E" for a, _ in pairs_plot]
ax.bar(labels, sims, color=cols)
ax.set_ylim(0, 1)
ax.set_ylabel("FastText kosinusi")
ax.set_title("Notanish shakllar va ularning soʻzlari (qizil — soxta oʻxshashlik)")
ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()

## 5. Topshiriqlar

1. **Sublinear TF.** TF-IDF ni $\text{tf}=1+\ln(\text{count})$ bilan (nolga teng boʻlmagan sanoqlar uchun) qoʻlda hisoblang va `TfidfVectorizer(sublinear_tf=True, tokenizer=tokenize, lowercase=False, token_pattern=None)` bilan solishtiring. Qaysi hujjatda oʻzgarish eng sezilarli va nima uchun?
2. **Oyna kengligi.** SGNS ni (2-boʻlim) `WINDOW = 1` va `WINDOW = 4` bilan oʻqiting va «shoʻrva» hamda «daftar» soʻzlarining beshta eng yaqin qoʻshnisini solishtiring. Qaysi oyna koʻproq «mavzuviy», qaysi biri koʻproq «grammatik» qoʻshnilar beradi?
3. **Yangi mavzu.** `corpus` ga transport haqida 6–8 ta gap qoʻshing (avtobus, poyezd, mashina, bekat, yuradi…) va 3-boʻlimdagi `Word2Vec` ni qayta oʻqiting. Yangi soʻzlar bir-birining qoʻshnisiga aylanadimi? `epochs=300` oʻrniga `epochs=50` qoʻyilsa nima oʻzgaradi?
4. **FastText va morfologiya.** Korpusdagi soʻzlarning korpusda boʻlmagan beshta shaklini oʻylab toping («mushuklarimiz», «itlar», «daftarlarga», «onamga», «nonni») va har biri uchun FastText boʻyicha eng yaqin soʻzni chiqaring. FastText umumiy harflar tufayli adashadigan misol toping.
5. **N-gramma uzunligi.** FastText ni `min_n=2, max_n=3` va `min_n=4, max_n=6` bilan oʻqiting. «kitoblarimizdan» — «kitob» va «suvenir» — «suv» oʻxshashligi qanday oʻzgaradi? Tushuntiring.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing